# Caso Práctico: Análisis de Emisiones de CO₂ Per Cápita

## Contexto

Las emisiones de **dióxido de carbono (CO₂)** son el principal gas de efecto invernadero producido por actividades humanas: combustión de combustibles fósiles, deforestación y procesos industriales. Comprender su distribución geográfica y evolución temporal es clave para diseñar políticas climáticas efectivas.

Este cuaderno analiza datos del **Banco Mundial** que miden las emisiones de CO₂ per cápita (en toneladas métricas por habitante) para países de todo el mundo entre **1960 y 2024**, enriquecidos con metadatos socioeconómicos como la región geográfica y el grupo de ingresos.

## Objetivos del análisis

1. Explorar la estructura y calidad de los datos.
2. Visualizar la distribución geográfica de las emisiones mediante mapas coropléticos.
3. Analizar la relación entre nivel de ingresos y emisiones de CO₂.
4. Examinar las tendencias temporales por región.

## Estructura del cuaderno

| Sección | Técnica de visualización | Pregunta clave |
|---------|--------------------------|----------------|
| Carga y exploración | — | ¿Qué cubre el dataset? |
| Calidad de datos | Estadísticas descriptivas | ¿Es fiable y completo? |
| Mapa coroplético (CO₂) | `px.choropleth` | ¿Qué países emiten más? |
| Mapa coroplético (ingresos) | `px.choropleth` | ¿Cómo se distribuye la riqueza? |
| Dispersión | `px.scatter` | ¿Influye el nivel de ingresos? |
| Caja y bigotes | `px.box` | ¿Cómo varía la distribución intra-grupo? |
| Histograma | `px.histogram` | ¿Qué forma tiene la distribución global? |
| Serie temporal | `px.line` | ¿Cómo ha evolucionado por región? |
| Mapa de calor | `px.imshow` | ¿Qué patrones espacio-temporales hay? |

> **Fuente de datos:** Banco Mundial — [CO2 emissions (metric tons per capita)](https://data.worldbank.org/indicator/EN.ATM.CO2E.PC)

In [ ]:
%pip install pandas plotly nbformat --quiet

## 1. Importación de Librerías

| Librería | Versión mínima | Propósito |
|----------|---------------|-----------|
| `pandas` | ≥ 1.5 | Carga, limpieza y transformación de datos tabulares |
| `plotly.express` | ≥ 5.0 | Visualizaciones interactivas con API de alto nivel |

> **¿Por qué Plotly Express?** A diferencia de Matplotlib o Seaborn, Plotly genera gráficos interactivos por defecto: el usuario puede hacer zoom, filtrar por leyenda y explorar valores exactos al pasar el cursor.

In [ ]:
import pandas as pd
import plotly.express as px

## 2. Carga y Exploración Inicial de Datos

Se trabaja con dos archivos CSV complementarios:

| Archivo | Contenido | Formato |
|---------|-----------|---------|
| `co2-per-capita.csv` | Emisiones de CO₂ (ton/hab) por país y año | **Ancho**: un año por columna |
| `metadata-country.csv` | Región geográfica y grupo de ingresos por país | **Largo**: atributos por fila |

### Sobre el formato ancho

Los datos de CO₂ están en **formato ancho** donde cada columna corresponde a un año (1960–2024). Este formato facilita comparar países en un año concreto, pero requiere transformación (`melt`) para análisis de series temporales.

In [ ]:
# Carga de los datos de CO₂
co2_data = pd.read_csv('co2-per-capita.csv')

# Resumen del dataset
year_cols = [c for c in co2_data.columns if str(c).isdigit()]
print(f"Dimensiones del dataset : {co2_data.shape[0]} filas x {co2_data.shape[1]} columnas")
print(f"Anos cubiertos          : {year_cols[0]} - {year_cols[-1]} ({len(year_cols)} anos)")
print(f"Paises/territorios      : {co2_data['Country Name'].nunique()}")
print(f"\nMuestra de 5 paises con sus emisiones en 2020:")
muestra = co2_data[['Country Name', 'Country Code', '2020']].dropna(subset=['2020']).sample(5, random_state=42)
print(muestra.to_string(index=False))

co2_data.head(3)

In [ ]:
metadata = pd.read_csv('metadata-country.csv')

print(f"Dimensiones: {metadata.shape[0]} filas x {metadata.shape[1]} columnas")
print(f"\nDistribucion por Region geografica:")
print(metadata['Region'].value_counts().to_string())
print(f"\nDistribucion por Grupo de Ingresos:")
print(metadata['IncomeGroup'].value_counts().to_string())
print(f"\nPaises sin clasificacion de region: {metadata['Region'].isna().sum()}")

metadata.head()

## 3. Unión de Datasets y Análisis de Calidad

### Estrategia de unión

Se realiza un **`inner join`** entre ambos datasets usando `Country Code` (código ISO-3) como clave. Esto conserva únicamente los países presentes en **ambos** datasets, descartando regiones agregadas (p.ej. "Africa Eastern and Southern") que no tienen metadatos individuales.

### Análisis de cobertura temporal

Un reto importante de este dataset es la **cobertura temporal incompleta**: no todos los países tienen datos para todos los años. Las causas incluyen:
- Países surgidos o disueltos en distintos períodos (p.ej. tras el fin de la URSS)
- Limitaciones estadísticas en economías en desarrollo
- Datos de años recientes (2022–2024) aún no publicados por el Banco Mundial

In [ ]:
# Union de datos basada en la columna 'Country Code'
data_merged = pd.merge(co2_data, metadata, how='inner', left_on='Country Code', right_on='Country Code')

print(f"Registros en co2_data    : {len(co2_data):>4}")
print(f"Registros en metadata    : {len(metadata):>4}")
print(f"Registros tras inner join: {len(data_merged):>4}")

# Analisis de cobertura temporal por decada
print("\nCobertura de datos por decada (porcentaje de paises con datos):")
for year in ['1960', '1970', '1980', '1990', '2000', '2010', '2020']:
    if year in data_merged.columns:
        pct = data_merged[year].notna().mean() * 100
        barra = '#' * int(pct / 5)
        print(f"  {year}: {barra:<20} {pct:.1f}%")

# Estadisticas clave del año 2020
vals_2020 = data_merged['2020'].dropna()
print(f"\nEstadisticas globales CO2 per capita (2020):")
print(f"  Paises con datos : {len(vals_2020)}")
print(f"  Media            : {vals_2020.mean():.2f} ton/hab")
print(f"  Mediana          : {vals_2020.median():.2f} ton/hab")
print(f"  Maximo           : {vals_2020.max():.2f} ton/hab")
print(f"\nTop 5 emisores per capita (2020):")
top5 = data_merged.nlargest(5, '2020')[['Country Name', 'Region', '2020']].dropna()
print(top5.to_string(index=False))

data_merged.head(3)

## 4. Mapas Coropléticos: CO₂ per Cápita y Grupo de Ingresos

Un **mapa coroplético** asigna un color a cada polígono geográfico (país) según el valor de una variable, permitiendo identificar patrones espaciales de un vistazo.

### ¿Qué esperamos ver?

- **Países del Golfo Pérsico** (Qatar, Kuwait, EAU): emisiones per cápita extremadamente altas, impulsadas por la industria petrolífera y altos subsidios energéticos.
- **Norteamérica y Australia**: emisiones altas por dependencia histórica del carbón y el petróleo.
- **Europa Occidental**: emisiones moderadas, con descenso progresivo por la transición energética.
- **África Subsahariana y Asia Meridional**: emisiones bajas, reflejo de menor industrialización.

> **Nota sobre la escala de color del primer mapa**: El rango se limita a 25 ton/hab para evitar que los valores extremos compriman visualmente la escala. Los países con valores superiores aparecen en el color máximo de la paleta.

In [ ]:
# Mapa colorpletico de CO2 per capita en 2020
fig = px.choropleth(
    data_merged.dropna(subset=['2020']),
    locations="Country Code",
    color="2020",
    hover_name="Country Name",
    hover_data={"Region": True, "IncomeGroup": True, "Country Code": False},
    color_continuous_scale="brwnyl",
    range_color=[0, 25],  # Limita la escala: outliers >25 aparecen en el color maximo
    labels={"2020": "CO2 per capita (ton/hab)"},
    title="Emisiones de CO2 per Capita (2020)",
)
fig.update_geos(showcoastlines=True, coastlinecolor="Black", projection_type="natural earth")
fig.update_layout(
    margin={"r": 0, "t": 50, "l": 0, "b": 0},
    coloraxis_colorbar=dict(title="ton CO2/hab", ticksuffix=" t")
)
fig.show()

In [ ]:
# Exploracion de los grupos de ingresos disponibles
income_counts = data_merged['IncomeGroup'].value_counts()
print("Distribucion de paises por grupo de ingresos:")
for group, count in income_counts.items():
    print(f"  {group:<25}: {count} paises")
print(f"\nSin clasificacion de ingresos: {data_merged['IncomeGroup'].isna().sum()} paises")

In [ ]:
# Mapa colorpletico por grupo de ingresos (colores discretos, sin conversion numerica)
income_order = ['Low income', 'Lower middle income', 'Upper middle income', 'High income']
income_colors = {
    'Low income': '#aed6f1',
    'Lower middle income': '#5dade2',
    'Upper middle income': '#2471a3',
    'High income': '#1a5276',
}

fig = px.choropleth(
    data_merged.dropna(subset=['IncomeGroup']),
    locations="Country Code",
    color="IncomeGroup",
    hover_name="Country Name",
    hover_data={"Region": True, "Country Code": False},
    category_orders={"IncomeGroup": income_order},
    color_discrete_map=income_colors,
    title="Clasificacion por Grupo de Ingresos - Banco Mundial (2020)",
)
fig.update_geos(showcoastlines=True, coastlinecolor="Black", projection_type="natural earth")
fig.update_layout(
    margin={"r": 0, "t": 50, "l": 0, "b": 0},
    legend_title="Grupo de Ingresos"
)
fig.show()

### Barras comparativas: CO₂ promedio por grupo de ingresos

El gráfico de barras sintetiza en un solo vistazo la diferencia entre grupos de ingresos. Cada barra muestra la **media** de emisiones per cápita en 2020, y la etiqueta indica el número de países incluidos en cada grupo.

In [ ]:
# Comparacion de emisiones promedio de CO2 por grupo de ingresos (grafico de barras)
income_order = ['Low income', 'Lower middle income', 'Upper middle income', 'High income']
income_colors = {
    'Low income': '#aed6f1',
    'Lower middle income': '#5dade2',
    'Upper middle income': '#2471a3',
    'High income': '#1a5276',
}

co2_by_income = (
    data_merged
    .dropna(subset=['2020', 'IncomeGroup'])
    .groupby('IncomeGroup')['2020']
    .agg(Media='mean', Mediana='median', N_paises='count')
    .reindex(income_order)
    .reset_index()
)

fig = px.bar(
    co2_by_income,
    x='IncomeGroup',
    y='Media',
    color='IncomeGroup',
    text='N_paises',
    category_orders={'IncomeGroup': income_order},
    color_discrete_map=income_colors,
    labels={'IncomeGroup': 'Grupo de Ingresos', 'Media': 'CO2 per Capita Promedio (ton/hab)'},
    title='CO2 per Capita Promedio por Grupo de Ingresos (2020)',
)
fig.update_traces(texttemplate='n=%{text} paises', textposition='outside')
fig.update_layout(showlegend=False, yaxis_title='CO2 per Capita Promedio (ton/hab)')
fig.show()

## 5. Gráfico de Dispersión: CO₂ per Cápita vs. Grupo de Ingresos

El **gráfico de dispersión** posiciona cada país como un punto según su grupo de ingresos (eje X) y sus emisiones de CO₂ en 2020 (eje Y).

### Hipótesis a contrastar

> ¿Los países con mayores ingresos emiten más CO₂ per cápita?

Se espera una relación positiva, pero con **alta variabilidad intra-grupo**:
- Algunos países ricos han reducido significativamente sus emisiones mediante transición energética (p.ej. Suecia, Francia).
- Países productores de petróleo muestran valores extremos independientemente de su nivel de ingresos.
- Países emergentes de altos ingresos (Arabia Saudí, Qatar) elevan la media del grupo superior.

> Pase el cursor sobre los puntos para ver el nombre del país y su región geográfica.

In [ ]:
income_order = ['Low income', 'Lower middle income', 'Upper middle income', 'High income']

fig = px.scatter(
    data_merged.dropna(subset=['2020', 'IncomeGroup']),
    x='IncomeGroup',
    y='2020',
    color='IncomeGroup',
    category_orders={'IncomeGroup': income_order},
    labels={
        'IncomeGroup': 'Grupo de Ingresos',
        '2020': 'CO2 per Capita 2020 (ton/hab)'
    },
    title='Relacion entre CO2 per Capita y Grupo de Ingresos (2020)',
    hover_data=['Country Name', 'Region'],
)
fig.update_traces(marker=dict(size=10, opacity=0.7))
fig.update_layout(
    showlegend=True,
    xaxis_title='Grupo de Ingresos',
    yaxis_title='CO2 per Capita (ton/hab)'
)
fig.show()

## 6. Gráfico de Caja y Bigotes: Distribución de CO₂ por Grupo de Ingresos

El **diagrama de caja** resume estadísticamente la distribución de emisiones dentro de cada grupo de ingresos:

| Elemento | Significado estadístico |
|---------|------------------------|
| Línea central de la caja | Mediana (percentil 50) |
| Borde inferior de la caja | Percentil 25 (Q1) |
| Borde superior de la caja | Percentil 75 (Q3) |
| Bigote inferior | Q1 − 1.5 × IQR (donde IQR = Q3 − Q1) |
| Bigote superior | Q3 + 1.5 × IQR |
| Puntos fuera de los bigotes | Outliers (valores atípicos) |

La opción `points='all'` superpone todos los puntos individuales sobre la caja, lo que permite ver la distribución real de los datos y no solo el resumen estadístico.

In [ ]:
income_order = ['Low income', 'Lower middle income', 'Upper middle income', 'High income']

fig = px.box(
    data_merged.dropna(subset=['2020', 'IncomeGroup']),
    x='IncomeGroup',
    y='2020',
    points='all',
    color='IncomeGroup',
    category_orders={'IncomeGroup': income_order},
    labels={
        'IncomeGroup': 'Grupo de Ingresos',
        '2020': 'CO2 per Capita (ton/hab)'
    },
    title='Distribucion de CO2 per Capita por Grupo de Ingresos (2020)',
    hover_data=['Country Name'],
)
fig.update_traces(marker=dict(size=5, opacity=0.6))
fig.update_layout(
    showlegend=False,
    xaxis_title='Grupo de Ingresos',
    yaxis_title='CO2 per Capita (ton/hab)'
)
fig.show()

## 7. Histograma: Distribución Global de CO₂ per Cápita

El **histograma** agrupa los países en intervalos de emisión (bins) y muestra cuántos países caen en cada rango. Coloreando por grupo de ingresos se puede ver cómo cada grupo contribuye a la distribución global.

### Conceptos estadísticos a observar

- **Sesgo positivo (cola derecha)**: La mayoría de países tiene emisiones bajas, pero unos pocos presentan valores muy altos. Esto hace que la **media > mediana**.
- **Distribución multimodal**: Posibles picos separados correspondientes a distintos grupos de ingresos.
- **Líneas de referencia**: La línea discontinua marca la mediana y la punteada la media, para contextualizar la asimetría de la distribución.

In [ ]:
vals_2020 = data_merged['2020'].dropna()
media = vals_2020.mean()
mediana = vals_2020.median()

print(f"Media global (2020)  : {media:.2f} ton/hab")
print(f"Mediana global (2020): {mediana:.2f} ton/hab")
print(f"-> La mediana ({mediana:.2f}) < media ({media:.2f}): distribucion con sesgo positivo (cola derecha)")

income_order = ['Low income', 'Lower middle income', 'Upper middle income', 'High income']

fig = px.histogram(
    data_merged.dropna(subset=['2020', 'IncomeGroup']),
    x='2020',
    nbins=30,
    color='IncomeGroup',
    barmode='overlay',
    opacity=0.7,
    category_orders={'IncomeGroup': income_order},
    labels={'2020': 'CO2 per Capita (ton/hab)', 'IncomeGroup': 'Grupo de Ingresos'},
    title='Distribucion de CO2 per Capita por Grupo de Ingresos (2020)',
)
fig.add_vline(
    x=mediana, line_dash='dash', line_color='red',
    annotation_text=f'Mediana: {mediana:.1f}', annotation_position='top right'
)
fig.add_vline(
    x=media, line_dash='dot', line_color='blue',
    annotation_text=f'Media: {media:.1f}', annotation_position='top left'
)
fig.update_layout(
    xaxis_title='CO2 per Capita (ton/hab)',
    yaxis_title='Numero de Paises',
    legend_title='Grupo de Ingresos',
    bargap=0.05
)
fig.show()

## 8. Transformación: Formato Ancho → Formato Largo

Para analizar **series temporales** con Plotly, necesitamos reformatear los datos de formato ancho a formato largo usando `pd.DataFrame.melt()`.

**Formato ancho** (actual):

| Country Name | 1960 | 1970 | 1980 | ... |
|-------------|------|------|------|-----|
| Spain | 1.2 | 3.4 | 5.6 | ... |
| Germany | 8.1 | 9.2 | 10.1 | ... |

**Formato largo** (resultado del `melt`):

| Country Name | Year | co2_per_capita |
|-------------|------|----------------|
| Spain | 1960 | 1.2 |
| Spain | 1970 | 3.4 |
| Germany | 1960 | 8.1 |

El filtro `co2_per_capita <= 4000` elimina regiones agregadas (p.ej. "Africa Eastern and Southern") que aparecen con valores calculados de forma diferente a los países individuales.

In [ ]:
# Detectar columnas id disponibles (manejo robusto de columnas opcionales)
base_id_vars = ["Country Name", "Country Code", "Indicator Name", "Indicator Code",
                "Region", "IncomeGroup", "TableName"]
optional_id_vars = ["Unnamed: 69", "IncomeGroupNum"]
id_vars = base_id_vars + [c for c in optional_id_vars if c in data_merged.columns]

# Transformar de formato ancho a formato largo
data_long = data_merged.melt(
    id_vars=id_vars,
    var_name='Year',
    value_name='co2_per_capita'
)

# Convertir 'Year' a numerico y eliminar filas invalidas
data_long['Year'] = pd.to_numeric(data_long['Year'], errors='coerce')
data_long = data_long.dropna(subset=['co2_per_capita'])

# Filtrar valores extremos de regiones agregadas
data_long_filtrado = data_long[data_long["co2_per_capita"] <= 4000]

print(f"Filas en formato largo: {len(data_long_filtrado):,}")
print(f"Anos cubiertos        : {int(data_long_filtrado['Year'].dropna().min())} - {int(data_long_filtrado['Year'].dropna().max())}")
print(f"Paises representados  : {data_long_filtrado['Country Name'].nunique()}")
data_long_filtrado.head()

## 9. Gráfico de Líneas: Evolución Temporal del CO₂ por Región

El gráfico de líneas muestra la evolución de las **emisiones promedio de CO₂** por región a lo largo del tiempo, permitiendo identificar tendencias globales y comparar trayectorias regionales.

### Patrones esperados

| Región | Tendencia esperada |
|--------|-------------------|
| América del Norte | Alta histórica, estabilización reciente |
| Europa & Asia Central | Alta histórica, descenso gradual desde 1990 |
| Asia Oriental & Pacífico | Fuerte crecimiento desde 1990 por industrialización |
| Oriente Medio & Norte de África | Crecimiento sostenido por economías petroleras |
| África Subsahariana | Bajas emisiones, crecimiento lento |
| Asia Meridional | Bajas emisiones, ligero crecimiento reciente |

> El modo `hovermode='x unified'` muestra los valores de todas las regiones al mismo tiempo al pasar el cursor sobre un año.

In [ ]:
# Media de CO2 por region y ano
mean_co2_by_region = data_long_filtrado.groupby(['Year', 'Region'], as_index=False)['co2_per_capita'].mean()

fig = px.line(
    mean_co2_by_region,
    x='Year',
    y='co2_per_capita',
    color='Region',
    markers=True,
    labels={
        'co2_per_capita': 'CO2 per Capita Promedio (ton/hab)',
        'Year': 'Ano',
        'Region': 'Region'
    },
    title='Evolucion del CO2 per Capita Promedio por Region (1960-2024)',
)
fig.update_traces(marker=dict(size=4))
fig.update_layout(
    xaxis_title='Ano',
    yaxis_title='CO2 per Capita Promedio (ton/hab)',
    legend_title='Region',
    hovermode='x unified'
)
fig.show()

## 10. Mapa de Calor: Evolución del CO₂ por Región y Año

El **mapa de calor** combina la dimensión geográfica (región, eje Y) con la temporal (año, eje X) en una sola visualización, usando el color para codificar la intensidad de las emisiones.

### Ventajas frente al gráfico de líneas

| Aspecto | Gráfico de líneas | Mapa de calor |
|---------|------------------|---------------|
| Precisión de valores | Alta (escala continua) | Media (interpolada por color) |
| Comparación entre regiones | Media | Alta (visión simultánea) |
| Identificación de patrones | Buena para tendencias | Buena para cambios bruscos |
| Cobertura de datos | Visible como discontinuidades | Visible como celdas vacías |

> **Paleta `Viridis`**: Va de morado oscuro (valores bajos) a amarillo (valores altos). Es perceptualmente uniforme y accesible para personas con daltonismo.

In [ ]:
# Media por region y ano, pivotada para el mapa de calor
heatmap_data = data_long_filtrado.groupby(['Year', 'Region'], as_index=False)['co2_per_capita'].mean()
heatmap_data_pivot = heatmap_data.pivot(index='Region', columns='Year', values='co2_per_capita')

fig = px.imshow(
    heatmap_data_pivot,
    aspect='auto',
    labels=dict(x='Ano', y='Region', color='CO2 per Capita (ton/hab)'),
    title='Evolucion del CO2 per Capita Promedio por Region (Mapa de Calor)',
    color_continuous_scale='Viridis',
)
fig.update_layout(
    xaxis_title='Ano',
    yaxis_title='Region',
    coloraxis_colorbar=dict(title='ton/hab')
)
fig.show()

## Conclusiones del Análisis

### Hallazgos principales

1. **Desigualdad global persistente**: Los países de ingresos altos emiten sistemáticamente más CO₂ per cápita que los de bajos ingresos, aunque existe alta variabilidad dentro de cada grupo.

2. **Los países del Golfo son outliers extremos**: Qatar, Kuwait y Emiratos Árabes presentan las emisiones per cápita más altas del mundo, impulsadas por la industria petrolífera y subsidios energéticos masivos.

3. **Fuerte crecimiento en Asia Oriental**: La región ha experimentado el mayor aumento de emisiones desde 1990, coincidiendo con la industrialización acelerada de China y otros países emergentes.

4. **Señales de descarbonización en Europa**: Europa y América del Norte muestran estabilización o ligera reducción en las últimas décadas, posiblemente relacionada con la transición hacia energías renovables y mejoras de eficiencia.

5. **Sesgo positivo en la distribución global**: La mayoría de países concentra emisiones bajas; la cola derecha la forman un puñado de países muy emisores que elevan la media muy por encima de la mediana.

6. **Cobertura incompleta en años recientes**: La disponibilidad de datos cae notablemente para 2022–2024, lo que limita el análisis de las tendencias más recientes.

### Próximos pasos sugeridos

- Correlacionar las emisiones con el PIB per cápita (curva de Kuznets ambiental).
- Analizar países con alta renta pero bajas emisiones (energía nuclear o renovable).
- Incorporar datos de emisiones **absolutas** (no per cápita) para ver el peso real de cada país.